# 集成算法（Ensemble Learning）学习笔记

---

## 目录

1. [集成学习概述](#1-集成学习概述)
2. [Bagging：并行取平均](#2-bagging并行取平均)
3. [随机森林：Bagging 的集大成者](#3-随机森林bagging-的集大成者)
4. [Boosting：串行加权强化](#4-boosting串行加权强化)
5. [AdaBoost：关注分错的样本](#5-adaboost关注分错的样本)
6. [Stacking：堆叠刷结果](#6-stacking堆叠刷结果)
7. [三大方法对比与总结](#7-三大方法对比与总结)

---

## 1. 集成学习概述

### 1.1 核心思想

> 课件金句：「让机器学习效果更好，单个不行，**群殴走起**。」

集成学习（Ensemble Learning）通过**组合多个模型**来提升整体性能，通常比任何单一模型都强。

### 1.2 三大方法总览

| 方法 | 核心思想 | 训练方式 | 典型代表 |
|:---|:---|:---|:---|
| **Bagging** | 多个分类器**取平均** | 并行 | 随机森林 |
| **Boosting** | 从弱学习器加强，**加权训练** | 串行 | AdaBoost、XGBoost |
| **Stacking** | 聚合多个模型，**分阶段** | 分阶段 | 堆叠各种模型 |

### 1.3 为什么集成有效？

- **Bagging**：降低**方差**（variance）——多个模型取平均，减少个别模型的过拟合
- **Boosting**：降低**偏差**（bias）——串行纠正错误，逐步逼近真实规律
- **Stacking**：融合不同模型的优势，提升精度

---

## 2. Bagging：并行取平均

### 2.1 全称与思想

**Bagging = Bootstrap Aggregation（自举聚合）**

> 说白了就是**并行训练一堆分类器**，最后投票或取平均。

### 2.2 核心机制：Bootstrap 采样

- **Bootstrap（有放回抽样）**：从原始数据中**有放回地**随机抽取多个子集
- 每个子集大小与原集相同，但因有放回，会有重复样本
- 用每个子集独立训练一个基模型

```
原始数据集 D
   ↓ Bootstrap 有放回抽样
子集1 → 模型1 ┐
子集2 → 模型2 ├─→ 投票/取平均 → 最终结果
子集3 → 模型3 ┘
```

### 2.3 为什么能降低方差

每个基模型在不同数据子集上训练，**相互独立、差异大**。对它们的预测取平均，能"抵消"各自的随机误差，使整体更稳定、泛化更好。

---

## 3. 随机森林：Bagging 的集大成者

### 3.1 什么是随机森林

> **森林 = 很多个决策树并行放在一起；随机 = 数据采样随机 + 特征选择随机。**

随机森林是 Bagging + 决策树的最典型代表。

### 3.2 两重随机性（核心创新）

| 随机性 | 做法 | 目的 |
|:---|:---|:---|
| **数据采样随机** | Bootstrap 有放回抽样 | 每棵树用不同数据 |
| **特征选择随机** | 每个节点从随机特征子集中选最优 | 每棵树的切分不同 |

> 课件强调：「由于二重随机性，每棵树基本不一样，最终结果也不一样。**随机是为了保证泛化能力——如果树都一样，那就没意义了。**」

### 3.3 构造树模型的流程

```
1. Bootstrap 有放回抽样，得到多个数据子集
2. 每个子集训练一棵决策树，但节点分裂时：
   - 随机选取一部分特征（而非全部）
   - 在选中的特征里找最优切分
3. 多棵树投票（分类）或取平均（回归）
```

### 3.4 随机森林的优势

- 能处理**很高维度**（feature 很多）的数据，**不用做特征选择**
- 训练后能给出**哪些 feature 更重要**（特征重要性）
- 可**可视化展示**，便于分析
- 容易**并行化**，速度快

### 3.5 树的数量如何选择？

> 「理论上越多的树效果越好，但实际超过一定数量后就上下浮动了。」

- 树太少：欠拟合，效果差
- 树太多：效果不再提升，只增加计算开销
- 做法：画"树数量 vs 误差"曲线，选趋于平稳的位置

### 3.6 为什么 KNN 不适合 Bagging？

> 课件指出 KNN 很难通过随机让泛化能力变强。

原因：KNN 本身对数据扰动敏感，且 Bootstrap 采样无法有效降低它的方差，"随机"并不能带来明显收益。

---

## 4. Boosting：串行加权强化

### 4.1 核心思想

> 「从弱学习器开始加强，通过加权来进行训练——**加入一棵树，要比原来强**。」

与 Bagging 的并行独立不同，Boosting 是**串行**的：每个新模型都**关注前一个模型犯的错**，逐步修正。

### 4.2 关键概念：弱学习器与强学习器

| 概念 | 含义 |
|:---|:---|
| **弱学习器** | 只比随机猜测略好的模型（如浅层决策树） |
| **强学习器** | 通过串行组合多个弱学习器，最终形成的高精度模型 |

> Boosting 的思路：**"三个臭皮匠，顶个诸葛亮"**——把多个弱学习器串起来，最终变成强学习器。

### 4.3 Boosting 的典型代表

| 算法 | 特点 |
|:---|:---|
| **AdaBoost** | 根据前一次分类效果调整数据权重（经典入门） |
| **GBDT** | 梯度提升，用残差拟合 |
| **XGBoost** | GBDT 的高效工程实现，竞赛神器 |
| **LightGBM** | 更快的梯度提升框架 |

---

## 5. AdaBoost：关注分错的样本

### 5.1 核心思想

> 「Adaboost 根据**前一次的分类效果调整数据权重**。最终结果：每个分类器根据自身准确性确定权重，再合体。**如果某个数据这次分错了，下次就给它更大的权重。**」

### 5.2 工作流程

```
1. 初始化：每个样本权重相同
2. 迭代训练弱分类器：
   a. 用当前权重训练一个弱分类器
   b. 计算该分类器的错误率 → 确定其话语权（权重）
   c. 提高分错样本的权重，降低分对样本的权重
3. 重复步骤 2，直到达到指定轮数
4. 加权投票：各弱分类器按权重合体 → 强分类器
```

**形象化（课件原文）：**

```
每一次切一刀！（弱分类器只比随机猜好一点）
  ↓ 关注上次切错的
最终合在一起（加权投票）
  ↓
弱分类器这就升级了！
```

### 5.3 关键机制

| 机制 | 说明 |
|:---|:---|
| **样本权重调整** | 分错的样本权重提高，让下一轮更关注它们 |
| **分类器权重** | 准确率高的分类器话语权大 |
| **加权投票** | 最终结果由各分类器按权重合并 |

---

## 6. Stacking：堆叠刷结果

### 6.1 核心思想

> 「堆叠：很暴力，拿来一堆直接上（各种分类器都来了）。可以堆叠 KNN、SVM、RF 等。**为了刷结果，不择手段！**」

Stacking 不是固定一种基模型，而是**混合多种不同类型的模型**，用它们的结果再训练一个元模型。

### 6.2 分阶段流程

```
第一阶段：训练多个不同基模型
  KNN → 预测结果1 ┐
  SVM → 预测结果2 ├─→ 作为新特征
  RF  → 预测结果3 ┘
           ↓
第二阶段：用第一阶段的输出训练元模型
  元模型 → 最终预测
```

### 6.3 特点

| 优点 | 缺点 |
|:---|:---|
| 确实能**提升准确率** | **速度是个问题**（训练多个模型 + 元模型） |
| 融合不同类型模型的优势 | 容易过拟合（需注意） |
| 竞赛与论文神器 | 实现和调参较复杂 |

> 课件总结：「集成算法是竞赛与论文神器，当我们更关注结果时不妨来试试！」

---

## 7. 三大方法对比与总结

### 7.1 对比表

| 维度 | Bagging | Boosting | Stacking |
|:---|:---|:---|:---|
| **训练方式** | 并行 | 串行 | 分阶段 |
| **基模型关系** | 独立 | 依赖（纠正前者错误） | 异构混合 |
| **核心目标** | 取平均 | 加权强化 | 堆叠融合 |
| **主要降低** | 方差（防过拟合） | 偏差（提精度） | 提精度 |
| **典型代表** | 随机森林 | AdaBoost、XGBoost | 多模型堆叠 |
| **速度** | 快（可并行） | 较慢（串行） | 慢（多模型） |

### 7.2 一句话总结

| 方法 | 一句话 |
|:---|:---|
| **Bagging** | 并行训练一堆模型取平均，降低方差，典型是随机森林 |
| **Boosting** | 串行训练，每个新模型纠正前一个的错误，降低偏差 |
| **Stacking** | 混合多种模型，分阶段堆叠，刷精度 |

### 7.3 选择建议

```
防过拟合、数据维度高？ → Bagging（随机森林）
追求精度、可接受串行？ → Boosting（XGBoost）
竞赛刷榜、不care速度？ → Stacking
```

---